<a href="https://colab.research.google.com/github/xCrisda/Taller2ColabIA/blob/main/Taller2IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea: A* (A-Star), Heurística y Teoría de Juegos en Python

**Nombre:** Cristian David García Valderrama  
**Fecha:** 16 de marzo de 2026  

En este notebook se desarrolla la investigación y la implementación en Python de los temas solicitados:
1. Algoritmo A* (A-Star)
2. Heurística
3. Teoría de juegos

# 1. A* (A-Star)

## 1.1 Teoría

El algoritmo **A\*** (A-Star) es un algoritmo de búsqueda informada utilizado para encontrar el **camino más corto** entre un punto inicial y un punto objetivo dentro de un grafo o una cuadrícula.

Es uno de los algoritmos más importantes en inteligencia artificial y se utiliza en áreas como:

- Videojuegos (búsqueda de rutas)
- Robótica
- Navegación
- Sistemas de mapas
- Inteligencia artificial

A* combina dos elementos fundamentales:

- **g(n):** el costo real recorrido desde el inicio hasta el nodo actual.
- **h(n):** una estimación del costo restante hasta llegar a la meta (heurística).

La fórmula principal del algoritmo es:

**f(n) = g(n) + h(n)**

Donde:
- **f(n)** representa el costo total estimado.
- El algoritmo siempre prioriza el nodo con menor valor de `f(n)`.

Gracias a esta combinación, A* puede encontrar rutas eficientes evitando explorar caminos innecesarios.

## 1.2 Código en Python y visualización del recorrido

A continuación, se presenta una implementación del algoritmo A* en Python sobre una cuadrícula (grid).  
El objetivo es encontrar el mejor camino desde una posición inicial hasta una meta, evitando obstáculos.

La visualización mostrará:

- **S** = posición inicial
- **G** = meta (goal)
- **#** = obstáculos
- **\*** = recorrido final encontrado

Además, se incluye una animación paso a paso del recorrido para observar cómo avanza la ruta encontrada.

In [ ]:
import heapq
import matplotlib.pyplot as plt
import numpy as np
import time
from IPython.display import clear_output

# =========================
# FUNCIONES DEL ALGORITMO A*
# =========================

def heuristica(a, b):
    # Distancia Manhattan
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def vecinos(pos, grid):
    movimientos = [(-1,0), (1,0), (0,-1), (0,1)]  # arriba, abajo, izquierda, derecha
    resultado = []
    for dx, dy in movimientos:
        nx, ny = pos[0] + dx, pos[1] + dy
        if 0 <= nx < len(grid) and 0 <= ny < len(grid[0]) and grid[nx][ny] == 0:
            resultado.append((nx, ny))
    return resultado

def a_estrella(grid, inicio, meta):
    abiertos = []
    heapq.heappush(abiertos, (0, inicio))

    vino_de = {}
    g = {inicio: 0}
    f = {inicio: heuristica(inicio, meta)}

    while abiertos:
        actual = heapq.heappop(abiertos)[1]

        if actual == meta:
            camino = []
            while actual in vino_de:
                camino.append(actual)
                actual = vino_de[actual]
            camino.append(inicio)
            camino.reverse()
            return camino

        for vecino in vecinos(actual, grid):
            g_temp = g[actual] + 1

            if vecino not in g or g_temp < g[vecino]:
                vino_de[vecino] = actual
                g[vecino] = g_temp
                f[vecino] = g[vecino] + heuristica(vecino, meta)
                heapq.heappush(abiertos, (f[vecino], vecino))

    return None

# =========================
# FUNCIÓN PARA DIBUJAR EL TABLERO
# =========================

def dibujar_tablero(grid, inicio, meta, camino=None, paso=None):
    filas, cols = len(grid), len(grid[0])

    # Crear una matriz visual
    visual = np.zeros((filas, cols))

    # Obstáculos
    for i in range(filas):
        for j in range(cols):
            if grid[i][j] == 1:
                visual[i][j] = 1  # obstáculo

    # Camino parcial o completo
    if camino:
        if paso is not None:
            for x, y in camino[:paso]:
                if (x, y) != inicio and (x, y) != meta:
                    visual[x][y] = 2
        else:
            for x, y in camino:
                if (x, y) != inicio and (x, y) != meta:
                    visual[x][y] = 2

    # Inicio y meta
    visual[inicio[0]][inicio[1]] = 3
    visual[meta[0]][meta[1]] = 4

    # Colores personalizados:
    # 0 = blanco (libre)
    # 1 = negro (obstáculo)
    # 2 = azul (camino)
    # 3 = verde (inicio)
    # 4 = rojo (meta)
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(["white", "black", "deepskyblue", "limegreen", "red"])

    plt.figure(figsize=(6,6))
    plt.imshow(visual, cmap=cmap)

    # Cuadrícula
    plt.xticks(np.arange(-0.5, cols, 1), [])
    plt.yticks(np.arange(-0.5, filas, 1), [])
    plt.grid(color='gray', linestyle='-', linewidth=1)

    # Etiquetas S y G
    plt.text(inicio[1], inicio[0], "S", ha='center', va='center', color='black', fontsize=14, fontweight='bold')
    plt.text(meta[1], meta[0], "G", ha='center', va='center', color='white', fontsize=14, fontweight='bold')

    plt.title("Algoritmo A* - Recorrido")
    plt.show()

# =========================
# EJEMPLO DE USO
# =========================

grid = [
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 0, 0, 1, 0, 1],
    [1, 1, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0]
]

inicio = (0, 0)
meta = (5, 5)

camino = a_estrella(grid, inicio, meta)

if camino:
    print("Camino encontrado:")
    print(camino)

    # Mostrar tablero final
    dibujar_tablero(grid, inicio, meta, camino)

    # Animación paso a paso
    print("Animación del recorrido:")
    time.sleep(1)

    for paso in range(1, len(camino) + 1):
        clear_output(wait=True)
        print("Camino encontrado:")
        print(camino)
        print(f"Paso {paso} de {len(camino)}")
        dibujar_tablero(grid, inicio, meta, camino, paso=paso)
        time.sleep(0.6)

else:
    print("No se encontró un camino.")

### Explicación del recorrido encontrado con A*

En esta visualización, el algoritmo A* busca un camino desde la posición inicial **S** (Start) hasta la meta **G** (Goal), evitando las celdas bloqueadas por obstáculos.

#### Elementos del tablero
- **S**: punto de inicio.
- **G**: punto objetivo o meta.
- **Celdas negras (#)**: obstáculos, no se pueden atravesar.
- **Celdas azules (*)**: recorrido encontrado por el algoritmo.
- **Celdas blancas**: espacios libres por donde sí se puede mover.

#### ¿Cómo se interpreta el recorrido?
El algoritmo comienza en **S** y evalúa las posibles rutas disponibles.  
A medida que avanza, evita los obstáculos y selecciona los movimientos que parecen más prometedores según su función de evaluación:

- **g(n)**: costo real acumulado desde el inicio.
- **h(n)**: estimación del costo restante hasta la meta (heurística).
- **f(n) = g(n) + h(n)**: valor total usado para decidir qué camino explorar.

En este ejemplo, A* encuentra un camino válido y eficiente desde **S** hasta **G**, mostrando cómo puede navegar en un entorno con obstáculos y llegar al objetivo utilizando una búsqueda informada.

#### ¿Qué muestra la animación?
La animación permite observar el recorrido **paso a paso**, resaltando cómo se construye la ruta final desde el inicio hasta la meta.  
Esto facilita entender visualmente el resultado del algoritmo y comprobar que el camino evita correctamente los obstáculos.

# 2. Heurística

## 2.1 ¿Qué es una heurística?

Una **heurística** es una técnica o estrategia que permite **estimar** cuál es la mejor opción para resolver un problema sin tener que probar todas las posibilidades.

En inteligencia artificial y en algoritmos de búsqueda, una heurística es una **función de estimación** que ayuda a decidir qué camino parece más prometedor para llegar a una meta.

En otras palabras, una heurística funciona como una **guía** que orienta al algoritmo para tomar decisiones más rápidas e inteligentes.

## 2.2 ¿Para qué sirve una heurística?

La heurística sirve para **reducir el tiempo de búsqueda** y hacer que un algoritmo sea más eficiente.

En lugar de explorar todas las rutas posibles, la heurística ayuda a priorizar aquellas que parecen acercarse más al objetivo.

### Beneficios principales:
- Reduce la cantidad de nodos o estados que se deben analizar.
- Acelera la búsqueda de soluciones.
- Hace más eficiente el uso de recursos computacionales.
- Permite encontrar soluciones óptimas o muy cercanas a la óptima en menos tiempo.

En el caso del algoritmo A*, la heurística es fundamental porque le permite estimar qué tan lejos está un nodo de la meta.

## 2.3 ¿Cómo funciona una heurística?

Una heurística asigna un valor estimado a cada posible estado o nodo.

Ese valor representa una **aproximación del costo restante** para llegar a la meta.

En el algoritmo A*, la heurística se representa normalmente como:

- **h(n)** = costo estimado desde el nodo actual hasta la meta.

A* combina esta estimación con el costo real ya recorrido:

- **g(n)** = costo real desde el inicio hasta el nodo actual.

Y utiliza la función:

- **f(n) = g(n) + h(n)**

Donde:

- **g(n)** indica cuánto costó llegar hasta ese punto.
- **h(n)** indica cuánto falta aproximadamente para llegar a la meta.
- **f(n)** es el valor total que A* usa para decidir qué nodo explorar primero.

Si la heurística está bien diseñada, el algoritmo puede encontrar soluciones más rápido.

## Ejemplos: Comparación entre heurística Manhattan y Euclidiana en A*

En este ejemplo se ejecuta el algoritmo A* dos veces sobre un tablero más complejo:

- una vez usando la **heurística Manhattan**
- otra vez usando la **heurística Euclidiana**

El objetivo es observar cómo cambia el comportamiento del algoritmo según la heurística utilizada.

Aunque ambas heurísticas pueden encontrar un camino óptimo, no siempre exploran la misma cantidad de nodos.  
Por eso, en este ejemplo se comparan:

- el **camino encontrado**
- la **longitud del camino**
- la **cantidad de nodos explorados**
- y una **visualización gráfica** del recorrido

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import numpy as np

# ==========================
# HEURÍSTICAS
# ==========================
def heuristica_manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def heuristica_euclidiana(a, b):
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

# ==========================
# A*
# ==========================
def a_estrella(tablero, inicio, meta, heuristica):
    filas, columnas = len(tablero), len(tablero[0])
    movimientos = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    abiertos = []
    heapq.heappush(abiertos, (0, inicio))

    vino_de = {}
    g = {inicio: 0}
    f = {inicio: heuristica(inicio, meta)}

    explorados = 0
    visitados = set()

    while abiertos:
        _, actual = heapq.heappop(abiertos)

        if actual in visitados:
            continue

        visitados.add(actual)
        explorados += 1

        if actual == meta:
            camino = []
            while actual in vino_de:
                camino.append(actual)
                actual = vino_de[actual]
            camino.append(inicio)
            camino.reverse()
            return camino, explorados, visitados

        for dx, dy in movimientos:
            vecino = (actual[0] + dx, actual[1] + dy)

            if 0 <= vecino[0] < filas and 0 <= vecino[1] < columnas:
                if tablero[vecino[0]][vecino[1]] == 1:
                    continue

                nuevo_g = g[actual] + 1

                if vecino not in g or nuevo_g < g[vecino]:
                    vino_de[vecino] = actual
                    g[vecino] = nuevo_g
                    f[vecino] = nuevo_g + heuristica(vecino, meta)
                    heapq.heappush(abiertos, (f[vecino], vecino))

    return None, explorados, visitados

# ==========================
# TABLERO MÁS COMPLEJO
# 0 = libre, 1 = obstáculo
# ==========================
tablero = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 1, 0],
    [0, 1, 0, 1, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 1, 0],
    [0, 1, 1, 0, 0, 0, 0, 0]
]

inicio = (0, 0)
meta = (7, 7)

# Ejecutar con ambas heurísticas
camino_m, explorados_m, visitados_m = a_estrella(tablero, inicio, meta, heuristica_manhattan)
camino_e, explorados_e, visitados_e = a_estrella(tablero, inicio, meta, heuristica_euclidiana)

# ==========================
# MOSTRAR RESULTADOS
# ==========================
print("=== COMPARACIÓN DE HEURÍSTICAS EN A* ===\n")

print("Heurística Manhattan:")
print("Camino:", camino_m)
print("Longitud del camino:", len(camino_m) if camino_m else "No encontrado")
print("Nodos explorados:", explorados_m)

print("\nHeurística Euclidiana:")
print("Camino:", camino_e)
print("Longitud del camino:", len(camino_e) if camino_e else "No encontrado")
print("Nodos explorados:", explorados_e)

# ==========================
# FUNCIÓN PARA GRAFICAR
# ==========================
def graficar(tablero, camino, visitados, titulo):
    filas, columnas = len(tablero), len(tablero[0])
    matriz = np.zeros((filas, columnas))

    for i in range(filas):
        for j in range(columnas):
            if tablero[i][j] == 1:
                matriz[i][j] = 1  # obstáculo

    for nodo in visitados:
        if tablero[nodo[0]][nodo[1]] == 0:
            matriz[nodo[0]][nodo[1]] = 2  # explorado

    if camino:
        for nodo in camino:
            matriz[nodo[0]][nodo[1]] = 3  # camino final

    matriz[inicio[0]][inicio[1]] = 4  # inicio
    matriz[meta[0]][meta[1]] = 5      # meta

    plt.figure(figsize=(6, 6))
    plt.imshow(matriz, cmap="viridis")
    plt.title(titulo)
    plt.grid(True)
    plt.xticks(range(columnas))
    plt.yticks(range(filas))
    plt.show()

# Graficar ambos resultados
graficar(tablero, camino_m, visitados_m, "A* con Heurística Manhattan")
graficar(tablero, camino_e, visitados_e, "A* con Heurística Euclidiana")

### Explicación del ejemplos

En este ejemplo se ejecuta el algoritmo A* dos veces sobre el mismo tablero con obstáculos, pero usando dos heurísticas diferentes:

- **Heurística Manhattan**
- **Heurística Euclidiana**

El objetivo es observar cómo cambia el comportamiento del algoritmo cuando la estimación de la distancia hacia la meta se calcula de forma distinta.

### ¿Qué hace el código?
El programa realiza los siguientes pasos:

1. Define un tablero con espacios libres y obstáculos.
2. Establece un punto inicial y una meta.
3. Ejecuta el algoritmo A* usando la **heurística Manhattan**.
4. Ejecuta nuevamente A* usando la **heurística Euclidiana**.
5. Muestra:
   - el camino encontrado,
   - la longitud del recorrido,
   - la cantidad de nodos explorados,
   - y una representación visual del resultado.

### ¿Qué se observa en este caso?
Aunque ambas heurísticas logran encontrar una ruta válida hasta la meta, el recorrido no es exactamente igual.

En la visualización se puede notar que:

- una de las rutas tiende a **bordear más los obstáculos**,
- mientras que la otra intenta avanzar por una ruta que parece **más directa** hacia la meta.

Esto ocurre porque, aunque ambas usan A*, cada heurística estima de forma diferente cuál es el mejor nodo para explorar en cada paso.

### ¿Por qué ocurre esto?
El algoritmo A* selecciona nodos usando la función:

**f(n) = g(n) + h(n)**

donde:

- **g(n)** representa el costo real desde el inicio hasta el nodo actual.
- **h(n)** representa una estimación de la distancia que falta para llegar a la meta.

La diferencia entre ambas ejecuciones está en la heurística **h(n)**:

- La **heurística Manhattan** calcula la distancia en una cuadrícula usando solo movimientos horizontales y verticales.  
  Por eso suele adaptarse mejor a tableros donde no se permiten diagonales y puede preferir rutas que bordean obstáculos de manera más ordenada.

- La **heurística Euclidiana** calcula la distancia en línea recta entre dos puntos.  
  Por eso tiende a favorecer nodos que parecen visualmente más cercanos a la meta, aunque luego deba rodear obstáculos para poder continuar.

### Conclusión
Este ejemplo demuestra que, aunque el algoritmo A* sea el mismo, la heurística utilizada puede cambiar el orden en que se exploran los nodos y, como consecuencia, también puede cambiar la forma del recorrido final.

En problemas de cuadrícula donde el movimiento se realiza solo en cuatro direcciones, la heurística de **Manhattan suele ser la más apropiada**, porque representa mejor el tipo de desplazamiento permitido.

# 3. Teoría de Juegos

## 3.1 ¿Qué es la teoría de juegos?

La teoría de juegos es una rama de las matemáticas y de la economía que estudia situaciones en las que dos o más participantes deben tomar decisiones, y el resultado para cada uno depende no solo de su propia elección, sino también de las decisiones de los demás.

En otras palabras, analiza escenarios de estrategia, competencia o cooperación, donde cada jugador busca obtener el mejor resultado posible según las acciones de los otros.

La teoría de juegos se aplica en problemas donde existen conflictos de interés, negociación, competencia, colaboración o toma de decisiones racionales.

## 3.2 ¿Para qué sirve la teoría de juegos?

La teoría de juegos sirve para analizar y comprender situaciones estratégicas en las que intervienen varios jugadores o agentes.

Permite:

- estudiar cómo se toman decisiones racionales,
- predecir posibles comportamientos de otros participantes,
- evaluar las consecuencias de diferentes estrategias,
- encontrar soluciones óptimas o equilibradas,
- y modelar conflictos o escenarios de cooperación.

Se utiliza en áreas como:

- economía,
- inteligencia artificial,
- negocios,
- política,
- biología,
- y ciencias de la computación.

## 3.3 ¿Cómo funciona la teoría de juegos?

La teoría de juegos funciona representando un problema como una interacción entre varios jugadores, donde cada uno tiene un conjunto de estrategias posibles y cada combinación de decisiones produce un resultado o ganancia, conocido como pago.

Los elementos principales son:

- **Jugadores:** quienes toman decisiones.
- **Estrategias:** las opciones disponibles para cada jugador.
- **Pagos o recompensas:** el resultado que obtiene cada jugador según las decisiones tomadas.

El análisis consiste en estudiar qué estrategia conviene elegir considerando que los demás jugadores también están tomando decisiones racionales.

En muchos casos, se busca identificar un punto de equilibrio, donde ningún jugador mejora su resultado cambiando su decisión de manera individual.  
Uno de los conceptos más importantes es el **equilibrio de Nash**, que representa una situación estable entre estrategias.

### 3.4 Ejemplo 1: Dilema del prisionero

El dilema del prisionero es uno de los ejemplos más conocidos de la teoría de juegos.

En este caso participan dos jugadores, y cada uno tiene dos posibles estrategias:

- **Cooperar**
- **Traicionar**

El resultado de cada jugador no depende únicamente de su propia decisión, sino también de la decisión del otro jugador.

Este ejemplo permite observar cómo, aunque cooperar mutuamente puede ser beneficioso para ambos, cada jugador puede verse tentado a elegir una estrategia individual que parezca mejor en el corto plazo.

In [ ]:
# ==========================================
# EJEMPLO 1: DILEMA DEL PRISIONERO (INTERACTIVO)
# ==========================================

# Matriz de pagos:
# (Jugador A, Jugador B)

pagos = {
    ("Cooperar", "Cooperar"): (3, 3),
    ("Cooperar", "Traicionar"): (0, 5),
    ("Traicionar", "Cooperar"): (5, 0),
    ("Traicionar", "Traicionar"): (1, 1)
}

def convertir_opcion(opcion):
    if opcion == 1:
        return "Cooperar"
    elif opcion == 2:
        return "Traicionar"
    else:
        return None

print("=== DILEMA DEL PRISIONERO INTERACTIVO ===\n")
print("Opciones:")
print("1 = Cooperar")
print("2 = Traicionar")

opcion_a = int(input("\nJugador A, elige tu opción (1 o 2): "))
accion_a = convertir_opcion(opcion_a)

opcion_b = int(input("Jugador B, elige tu opción (1 o 2): "))
accion_b = convertir_opcion(opcion_b)

if accion_a is None or accion_b is None:
    print("\n Opción inválida. Debes elegir 1 o 2.")
else:
    pago_a, pago_b = pagos[(accion_a, accion_b)]

    print("\n=== RESULTADO ===")
    print(f"Jugador A eligió: {accion_a}")
    print(f"Jugador B eligió: {accion_b}")
    print(f"Pago Jugador A: {pago_a}")
    print(f"Pago Jugador B: {pago_b}")

    print("\n=== INTERPRETACIÓN ===")
    if accion_a == "Cooperar" and accion_b == "Cooperar":
        print("Ambos cooperaron. Los dos obtienen una buena recompensa.")
    elif accion_a == "Traicionar" and accion_b == "Cooperar":
        print("Jugador A traicionó y obtuvo la mayor ganancia individual.")
    elif accion_a == "Cooperar" and accion_b == "Traicionar":
        print("Jugador B traicionó y obtuvo la mayor ganancia individual.")
    else:
        print("Ambos traicionaron. El resultado es peor que si ambos hubieran cooperado.")

### Explicación del ejemplo 1

En este ejemplo se implementa una versión interactiva del dilema del prisionero.

Cada jugador debe elegir una estrategia mediante un número:

- **1 = Cooperar**
- **2 = Traicionar**

El programa convierte esas opciones en decisiones reales dentro del juego y luego consulta una matriz de pagos previamente definida.

### ¿Qué hace el código?
El código realiza los siguientes pasos:

1. Define una matriz de pagos con las posibles combinaciones entre cooperar y traicionar.
2. Solicita al Jugador A y al Jugador B que ingresen su decisión mediante un número.
3. Convierte ese número en una estrategia válida.
4. Busca el resultado correspondiente en la matriz de pagos.
5. Muestra el pago de cada jugador.
6. Finalmente, interpreta automáticamente el resultado obtenido.

### ¿Qué se observa?
Se observa que el resultado depende de la combinación de decisiones de ambos jugadores.

Por ejemplo:

- si ambos cooperan, ambos reciben una buena recompensa,
- si uno coopera y el otro traiciona, el que traiciona obtiene la mayor ganancia,
- si ambos traicionan, ambos reciben un resultado menos favorable.

### Conclusión
Este ejemplo demuestra que en teoría de juegos las decisiones individuales afectan directamente a todos los participantes.

Además, permite ver cómo una elección racional para un jugador puede no ser la mejor opción para el grupo completo.

# 3.5 Ejemplo 2: Juego de coordinación

### 3.6 Ejemplo 2: Juego de coordinación

El juego de coordinación es un tipo de problema en teoría de juegos donde dos jugadores obtienen mejores resultados cuando logran tomar decisiones compatibles entre sí.

En este ejemplo, ambos jugadores deben elegir entre dos opciones:

- **Estrategia A**
- **Estrategia B**

La idea principal es que, si ambos eligen la misma estrategia, obtienen una mejor recompensa.  
Si eligen estrategias diferentes, el resultado es menos favorable.

# Este tipo de juego permite observar cómo, en muchos escenarios, no basta con tomar una decisión individualmente buena, sino que también es importante coordinarse con los demás.

In [ ]:
# ==========================================
# EJEMPLO 2: JUEGO DE COORDINACIÓN (INTERACTIVO)
# ==========================================

# Matriz de pagos:
# (Jugador A, Jugador B)

pagos = {
    ("A", "A"): (4, 4),
    ("A", "B"): (1, 1),
    ("B", "A"): (1, 1),
    ("B", "B"): (3, 3)
}

def convertir_opcion(opcion):
    if opcion == 1:
        return "A"
    elif opcion == 2:
        return "B"
    else:
        return None

print("=== JUEGO DE COORDINACIÓN INTERACTIVO ===\n")
print("Opciones:")
print("1 = Estrategia A")
print("2 = Estrategia B")

# Entrada del Jugador A
opcion_a = int(input("\nJugador A, elige tu opción (1 o 2): "))
accion_a = convertir_opcion(opcion_a)

# Entrada del Jugador B
opcion_b = int(input("Jugador B, elige tu opción (1 o 2): "))
accion_b = convertir_opcion(opcion_b)

# Validación
if accion_a is None or accion_b is None:
    print("\n Opción inválida. Debes elegir 1 o 2.")
else:
    pago_a, pago_b = pagos[(accion_a, accion_b)]

    print("\n=== RESULTADO ===")
    print(f"Jugador A eligió: Estrategia {accion_a}")
    print(f"Jugador B eligió: Estrategia {accion_b}")
    print(f"Pago Jugador A: {pago_a}")
    print(f"Pago Jugador B: {pago_b}")

    # Interpretación automática
    print("\n=== INTERPRETACIÓN ===")
    if accion_a == accion_b:
        if accion_a == "A":
            print("Ambos jugadores se coordinaron en la Estrategia A, obteniendo el mejor resultado posible.")
        else:
            print("Ambos jugadores se coordinaron en la Estrategia B, obteniendo un buen resultado.")
    else:
        print("Los jugadores no se coordinaron. Al elegir estrategias diferentes, ambos obtienen un resultado menor.")

### Explicación del ejemplo 2

En este ejemplo se implementa un juego de coordinación de forma interactiva.

Cada jugador debe elegir una estrategia mediante un número:

- **1 = Estrategia A**
- **2 = Estrategia B**

El programa convierte esas opciones en estrategias válidas y luego consulta una matriz de pagos.

### ¿Qué hace el código?
El código realiza los siguientes pasos:

1. Define una matriz de pagos con las posibles combinaciones entre las estrategias A y B.
2. Solicita al Jugador A y al Jugador B que ingresen su decisión.
3. Convierte cada número en una estrategia válida.
4. Busca el resultado correspondiente en la matriz de pagos.
5. Muestra el pago de cada jugador.
6. Interpreta automáticamente si hubo coordinación o no.

### ¿Qué se observa?
Se observa que:

- si ambos jugadores eligen la misma estrategia, obtienen un mejor resultado,
- si ambos eligen **A**, reciben la mejor recompensa posible,
- si ambos eligen **B**, también obtienen un buen resultado,
- si eligen estrategias distintas, ambos reciben un pago menor.

### Conclusión
Este ejemplo demuestra que en algunos juegos el beneficio no depende únicamente de la decisión individual, sino también de la capacidad de coordinarse con otro jugador.

La teoría de juegos permite analizar este tipo de situaciones estratégicas y entender cómo la cooperación o la coordinación pueden mejorar los resultados.

### 3.7 Ejemplo 3: Piedra, papel o tijera

El juego de piedra, papel o tijera es un ejemplo clásico que también puede analizarse desde la teoría de juegos.

En este tipo de juego, dos jugadores eligen simultáneamente una de tres estrategias posibles:

- **Piedra**
- **Papel**
- **Tijera**

Cada estrategia puede ganar, perder o empatar dependiendo de la elección del oponente:

- Piedra vence a tijera
- Tijera vence a papel
- Papel vence a piedra

Este ejemplo permite observar cómo las decisiones estratégicas generan distintos resultados y cómo, en algunos juegos, no existe una estrategia única que garantice siempre la victoria.

In [26]:
# ==========================================
# EJEMPLO 3: PIEDRA, PAPEL O TIJERA (INTERACTIVO)
# ==========================================

def convertir_opcion(opcion):
    if opcion == 1:
        return "Piedra"
    elif opcion == 2:
        return "Papel"
    elif opcion == 3:
        return "Tijera"
    else:
        return None

print("=== PIEDRA, PAPEL O TIJERA INTERACTIVO ===\n")
print("Opciones:")
print("1 = Piedra")
print("2 = Papel")
print("3 = Tijera")

# Entrada del Jugador A
opcion_a = int(input("\nJugador A, elige tu opción (1, 2 o 3): "))
accion_a = convertir_opcion(opcion_a)

# Entrada del Jugador B
opcion_b = int(input("Jugador B, elige tu opción (1, 2 o 3): "))
accion_b = convertir_opcion(opcion_b)

# Validación
if accion_a is None or accion_b is None:
    print("\n Opción inválida. Debes elegir 1, 2 o 3.")
else:
    print("\n=== RESULTADO ===")
    print(f"Jugador A eligió: {accion_a}")
    print(f"Jugador B eligió: {accion_b}")

    # Determinar ganador
    if accion_a == accion_b:
        print("Resultado: Empate")
        print("Ambos jugadores eligieron la misma estrategia.")
    elif (accion_a == "Piedra" and accion_b == "Tijera") or \
         (accion_a == "Tijera" and accion_b == "Papel") or \
         (accion_a == "Papel" and accion_b == "Piedra"):
        print("Resultado: Gana el Jugador A")
        print(f"{accion_a} vence a {accion_b}.")
    else:
        print("Resultado: Gana el Jugador B")
        print(f"{accion_b} vence a {accion_a}.")

=== PIEDRA, PAPEL O TIJERA INTERACTIVO ===

Opciones:
1 = Piedra
2 = Papel
3 = Tijera

Jugador A, elige tu opción (1, 2 o 3): 1
Jugador B, elige tu opción (1, 2 o 3): 2

=== RESULTADO ===
Jugador A eligió: Piedra
Jugador B eligió: Papel
Resultado: Gana el Jugador B
Papel vence a Piedra.


### Explicación del ejemplo 3

En este ejemplo se implementa una versión interactiva del juego piedra, papel o tijera.

Cada jugador debe elegir una estrategia mediante un número:

- **1 = Piedra**
- **2 = Papel**
- **3 = Tijera**

El programa convierte esos números en jugadas válidas y luego compara ambas elecciones para determinar el resultado.

### ¿Qué hace el código?
El código realiza los siguientes pasos:

1. Define una función que convierte un número en una estrategia válida.
2. Solicita al Jugador A y al Jugador B que ingresen su elección.
3. Valida que las opciones ingresadas sean correctas.
4. Compara las dos estrategias elegidas.
5. Determina si el resultado es empate o si alguno de los jugadores gana.
6. Muestra una explicación del por qué una estrategia vence a la otra.

### ¿Qué se observa?
Se observa que:

- si ambos jugadores eligen la misma opción, el resultado es un empate,
- si eligen opciones diferentes, una estrategia vence a la otra según las reglas del juego,
- no existe una única estrategia que siempre garantice ganar.

### Conclusión
Este ejemplo demuestra que en teoría de juegos existen situaciones donde el resultado depende directamente de la elección del oponente.

Además, permite entender que algunas decisiones estratégicas son relativas, es decir, una opción puede ser buena o mala dependiendo de la estrategia contraria.